# Supervised Learning: Linear and Logistic Regression

This notebook introduces the two foundational supervised learning algorithms:
- **Linear Regression** for continuous targets
- **Logistic Regression** for classification

We will use scikit-learn on real datasets, inspect model coefficients, and visualise decision boundaries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)

## 1. Linear Regression

The model assumes a linear relationship $y = X\beta + \varepsilon$.
We minimise the **ordinary least-squares** objective:
$$\hat{\beta} = \arg\min_\beta \|y - X\beta\|^2$$

In [ ]:
# Load the diabetes dataset (regression task)
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
feature_names = diabetes.feature_names

print(f"Features: {feature_names}")
print(f"Shape: {X.shape}, Target range: [{y.min():.0f}, {y.max():.0f}]")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Fit linear regression
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
print(f"R² score: {r2_score(y_test, y_pred):.3f}")
print(f"RMSE:     {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")

# Coefficient importance
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': lr.coef_})
coef_df = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=True).index)
coef_df.plot.barh(x='feature', y='coefficient', legend=False)
plt.title('Linear Regression Coefficients')
plt.tight_layout()
plt.show()

In [ ]:
# Residual plot
residuals = y_test - y_pred
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted')
plt.ylabel('Residual')
plt.title('Residual Plot')
plt.show()

## 2. Logistic Regression

For binary or multi-class classification, logistic regression models
$$P(y=1 \mid x) = \sigma(x^\top \beta) = \frac{1}{1+e^{-x^\top \beta}}$$

The loss is the **cross-entropy** (negative log-likelihood).

In [ ]:
# Load Iris dataset (multi-class classification)
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_iris_scaled, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

In [ ]:
# Fit logistic regression (one-vs-rest for multi-class)
logr = LogisticRegression(max_iter=500, multi_class='ovr', random_state=42)
logr.fit(X_train_i, y_train_i)

y_pred_i = logr.predict(X_test_i)
print(f"Accuracy: {accuracy_score(y_test_i, y_pred_i):.3f}")
print()
print(classification_report(y_test_i, y_pred_i, target_names=iris.target_names))

In [ ]:
# Decision boundary (2D projection on first two features)
from matplotlib.colors import ListedColormap

X2 = X_iris_scaled[:, :2]
logr2 = LogisticRegression(max_iter=500, random_state=42).fit(X2, y_iris)

h = 0.02
x_min, x_max = X2[:, 0].min() - 1, X2[:, 0].max() + 1
y_min, y_max = X2[:, 1].min() - 1, X2[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = logr2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
plt.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_light)
scatter = plt.scatter(X2[:, 0], X2[:, 1], c=y_iris, edgecolors='k', s=30)
plt.xlabel('Sepal length (scaled)')
plt.ylabel('Sepal width (scaled)')
plt.title('Logistic Regression Decision Boundaries')
plt.show()

## Key Takeaways

| Aspect | Linear Regression | Logistic Regression |
|--------|-------------------|---------------------|
| Task | Regression | Classification |
| Loss | MSE | Cross-entropy |
| Output | Real value | Probability |
| Regularisation | Ridge (L2), Lasso (L1) | Same options available |

**Next notebook:** Decision trees and ensemble methods.